#Import

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import date


#Get Data From Bronze

In [0]:
# Load product data from Azure Data Lake Storage in Parquet format
df = spark.read.format("parquet")\
        .option("inferSchema",True)\
        .load("abfss://bronze@storenapatdatalake.dfs.core.windows.net/crm_prd_info")
df.limit(10).display()

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
582,BI-RB-BK-R79Y-44,Road-350-W Yellow- 44,1083,R,2013-07-01,null
583,BI-RB-BK-R79Y-48,Road-350-W Yellow- 48,1083,R,2013-07-01,null
584,BI-RB-BK-R19B-58,Road-750 Black- 58,344,R,2013-07-01,null
585,BI-TB-BK-T18U-44,Touring-3000 Blue- 44,461,T,2013-07-01,null
586,BI-TB-BK-T18U-50,Touring-3000 Blue- 50,461,T,2013-07-01,null
587,BI-MB-BK-M38S-38,Mountain-400-W Silver- 38,420,M,2013-07-01,null
588,BI-MB-BK-M38S-40,Mountain-400-W Silver- 40,420,M,2013-07-01,null
589,BI-MB-BK-M38S-42,Mountain-400-W Silver- 42,420,M,2013-07-01,null
590,BI-MB-BK-M38S-46,Mountain-400-W Silver- 46,420,M,2013-07-01,null
591,BI-MB-BK-M18S-40,Mountain-500 Silver- 40,308,M,2013-07-01,null


#Data Transformation

##Trimming

In [0]:
# Remove leading and trailing spaces from all string columns
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

In [0]:
df.limit(10).display()

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
582,BI-RB-BK-R79Y-44,Road-350-W Yellow- 44,1083,R,2013-07-01,null
583,BI-RB-BK-R79Y-48,Road-350-W Yellow- 48,1083,R,2013-07-01,null
584,BI-RB-BK-R19B-58,Road-750 Black- 58,344,R,2013-07-01,null
585,BI-TB-BK-T18U-44,Touring-3000 Blue- 44,461,T,2013-07-01,null
586,BI-TB-BK-T18U-50,Touring-3000 Blue- 50,461,T,2013-07-01,null
587,BI-MB-BK-M38S-38,Mountain-400-W Silver- 38,420,M,2013-07-01,null
588,BI-MB-BK-M38S-40,Mountain-400-W Silver- 40,420,M,2013-07-01,null
589,BI-MB-BK-M38S-42,Mountain-400-W Silver- 42,420,M,2013-07-01,null
590,BI-MB-BK-M38S-46,Mountain-400-W Silver- 46,420,M,2013-07-01,null
591,BI-MB-BK-M18S-40,Mountain-500 Silver- 40,308,M,2013-07-01,null


##Product Key Parsing

In [0]:
# Extract category ID and standardize the product key
df = df.withColumn("cat_id",regexp_replace(substring(col("prd_key"),1,5),"-","_"))
df = df.withColumn("prd_key",substring(col("prd_key"),7,length(col("prd_key"))))
df.limit(10).display()

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,cat_id
582,BK-R79Y-44,Road-350-W Yellow- 44,1083,R,2013-07-01,null,BI_RB
583,BK-R79Y-48,Road-350-W Yellow- 48,1083,R,2013-07-01,null,BI_RB
584,BK-R19B-58,Road-750 Black- 58,344,R,2013-07-01,null,BI_RB
585,BK-T18U-44,Touring-3000 Blue- 44,461,T,2013-07-01,null,BI_TB
586,BK-T18U-50,Touring-3000 Blue- 50,461,T,2013-07-01,null,BI_TB
587,BK-M38S-38,Mountain-400-W Silver- 38,420,M,2013-07-01,null,BI_MB
588,BK-M38S-40,Mountain-400-W Silver- 40,420,M,2013-07-01,null,BI_MB
589,BK-M38S-42,Mountain-400-W Silver- 42,420,M,2013-07-01,null,BI_MB
590,BK-M38S-46,Mountain-400-W Silver- 46,420,M,2013-07-01,null,BI_MB
591,BK-M18S-40,Mountain-500 Silver- 40,308,M,2013-07-01,null,BI_MB


##Cleaning prd_cost

In [0]:
# Replace null product costs with zero
df = df.withColumn("prd_cost",coalesce(col("prd_cost"),lit(0)))
df.limit(10).display()

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,cat_id
582,BK-R79Y-44,Road-350-W Yellow- 44,1083,R,2013-07-01,null,BI_RB
583,BK-R79Y-48,Road-350-W Yellow- 48,1083,R,2013-07-01,null,BI_RB
584,BK-R19B-58,Road-750 Black- 58,344,R,2013-07-01,null,BI_RB
585,BK-T18U-44,Touring-3000 Blue- 44,461,T,2013-07-01,null,BI_TB
586,BK-T18U-50,Touring-3000 Blue- 50,461,T,2013-07-01,null,BI_TB
587,BK-M38S-38,Mountain-400-W Silver- 38,420,M,2013-07-01,null,BI_MB
588,BK-M38S-40,Mountain-400-W Silver- 40,420,M,2013-07-01,null,BI_MB
589,BK-M38S-42,Mountain-400-W Silver- 42,420,M,2013-07-01,null,BI_MB
590,BK-M38S-46,Mountain-400-W Silver- 46,420,M,2013-07-01,null,BI_MB
591,BK-M18S-40,Mountain-500 Silver- 40,308,M,2013-07-01,null,BI_MB


##Product Line Normalization

In [0]:
# Convert product line codes into descriptive labels
df = (
    df.withColumn("prd_line",
        when(upper(col("prd_line")) == "M", "Mountain")
         .when(upper(col("prd_line")) == "R", "Road")
         .when(upper(col("prd_line")) == "S", "Other Sales")
         .when(upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)
df.limit(10).display()

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,cat_id
582,BK-R79Y-44,Road-350-W Yellow- 44,1083,Road,2013-07-01,null,BI_RB
583,BK-R79Y-48,Road-350-W Yellow- 48,1083,Road,2013-07-01,null,BI_RB
584,BK-R19B-58,Road-750 Black- 58,344,Road,2013-07-01,null,BI_RB
585,BK-T18U-44,Touring-3000 Blue- 44,461,Touring,2013-07-01,null,BI_TB
586,BK-T18U-50,Touring-3000 Blue- 50,461,Touring,2013-07-01,null,BI_TB
587,BK-M38S-38,Mountain-400-W Silver- 38,420,Mountain,2013-07-01,null,BI_MB
588,BK-M38S-40,Mountain-400-W Silver- 40,420,Mountain,2013-07-01,null,BI_MB
589,BK-M38S-42,Mountain-400-W Silver- 42,420,Mountain,2013-07-01,null,BI_MB
590,BK-M38S-46,Mountain-400-W Silver- 46,420,Mountain,2013-07-01,null,BI_MB
591,BK-M18S-40,Mountain-500 Silver- 40,308,Mountain,2013-07-01,null,BI_MB


##Renaming Columns

In [0]:
# Rename source columns to business-friendly names
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)
df.limit(10).display()

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id
582,BK-R79Y-44,Road-350-W Yellow- 44,1083,Road,2013-07-01,null,BI_RB
583,BK-R79Y-48,Road-350-W Yellow- 48,1083,Road,2013-07-01,null,BI_RB
584,BK-R19B-58,Road-750 Black- 58,344,Road,2013-07-01,null,BI_RB
585,BK-T18U-44,Touring-3000 Blue- 44,461,Touring,2013-07-01,null,BI_TB
586,BK-T18U-50,Touring-3000 Blue- 50,461,Touring,2013-07-01,null,BI_TB
587,BK-M38S-38,Mountain-400-W Silver- 38,420,Mountain,2013-07-01,null,BI_MB
588,BK-M38S-40,Mountain-400-W Silver- 40,420,Mountain,2013-07-01,null,BI_MB
589,BK-M38S-42,Mountain-400-W Silver- 42,420,Mountain,2013-07-01,null,BI_MB
590,BK-M38S-46,Mountain-400-W Silver- 46,420,Mountain,2013-07-01,null,BI_MB
591,BK-M18S-40,Mountain-500 Silver- 40,308,Mountain,2013-07-01,null,BI_MB


##Create silver record_priority to filter out duplicate id

In [0]:
# Rank products by ID, prioritizing the latest start date
w = Window.partitionBy(df["product_id"]).orderBy(df["start_date"].desc_nulls_last())
df = df.withColumn("record_priority",row_number().over(w))

In [0]:
df.limit(5).display()

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id,record_priority
210,FR-R92B-58,HL Road Frame - Black- 58,0,Road,2003-07-01,null,CO_RF,1
210,FR-R92B-58,HL Road Frame - Black- 58,0,Road,2003-07-01,null,CO_RF,2
211,FR-R92R-58,HL Road Frame - Red- 58,0,Road,2003-07-01,null,CO_RF,1
211,FR-R92R-58,HL Road Frame - Red- 58,0,Road,2003-07-01,null,CO_RF,2
212,HL-U509-R,Sport-100 Helmet- Red,12,Other Sales,2011-07-01,2007-12-28,AC_HE,1


In [0]:
# Keep only the highest-priority record for each product
df = df.filter(col("record_priority") == 1)

#Data Writing

In [0]:
# Save cleaned product data in Parquet format
df.write.format("parquet")\
    .mode("overwrite")\
    .option("path","abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_prd_info")\
    .save()

In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_prd_info` LIMIT 10

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id,record_priority
210,FR-R92B-58,HL Road Frame - Black- 58,0,Road,2003-07-01,null,CO_RF,1
211,FR-R92R-58,HL Road Frame - Red- 58,0,Road,2003-07-01,null,CO_RF,1
212,HL-U509-R,Sport-100 Helmet- Red,12,Other Sales,2011-07-01,2007-12-28,AC_HE,1
213,HL-U509-R,Sport-100 Helmet- Red,14,Other Sales,2012-07-01,2008-12-27,AC_HE,1
214,HL-U509-R,Sport-100 Helmet- Red,13,Other Sales,2013-07-01,null,AC_HE,1
215,HL-U509,Sport-100 Helmet- Black,12,Other Sales,2011-07-01,2007-12-28,AC_HE,1
216,HL-U509,Sport-100 Helmet- Black,14,Other Sales,2012-07-01,2008-12-27,AC_HE,1
217,HL-U509,Sport-100 Helmet- Black,13,Other Sales,2013-07-01,null,AC_HE,1
218,SO-B909-M,Mountain Bike Socks- M,3,Mountain,2011-07-01,2007-12-28,CL_SO,1
219,SO-B909-L,Mountain Bike Socks- L,3,Mountain,2011-07-01,2007-12-28,CL_SO,1


In [0]:
%sql
SELECT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_prd_info` WHERE record_priority > 1 ORDER BY record_priority DESC LIMIT 10

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id,record_priority
